# Pipeline 0 — Data preparation

This notebook reads the raw Slovak firm panel (`firmy_dataset.csv`) and produces
three model-ready datasets:

- `data_logit.csv` / `data_logit_strict.csv` — features for the logistic
  regression benchmark (engineered ratios + raw accounting items + macro + NACE
  industry dummies + non-financial controls).
- `data_ebm.csv` — features for the Explainable Boosting Machine, including
  lag-1 / lag-2 levels and first/second differences. Missing values are encoded
  as the numeric sentinel `-999`; inspect its fitted effect separately.
- `data_rnn.csv` — sequence-ready features for the BiLSTM, restricted to firms
  with at least two consecutive years of data. Missing values are imputed with
  the training-period median.

Key design choices:

- SME filter on employees / total assets / turnover, with a `soft_missing` mode
  that keeps observations whose employee count or turnover is missing as long
  as the remaining size criteria are satisfied.
- Winsorization at the 5th / 95th percentiles, with cut-offs computed
  exclusively on the training years 2013–2016 to avoid look-ahead leakage.
- Macroeconomic controls (HDP growth, unemployment, ECB rate, Euribor, etc.)
  joined at the year level.
- Industry dummies derived from the textual SK NACE category.
- Out-of-time split: train 2013–2016, validation 2017, test 2018–2019.

## Class-imbalance strategy used across the pipeline

The bankruptcy rate in the Slovak firm-year panel is approximately 0.21 %. The thesis discusses several remedies for class imbalance, but the exported CSV files remain **unresampled**. Class-imbalance handling is applied only inside the downstream model notebooks.

| Pipeline | Strategy implemented later | Rationale |
|---|---|---|
| `01_logit_benchmark` | No resampling and no sample weights | Econometric benchmark in the spirit of Káčer & Alexy (2025); coefficient and odds-ratio interpretation remain direct. |
| `03_ebm_pipeline` | Strategy tournament: sector-stratified undersampling (`1:1`, `1:3`, `1:5`) versus class/sample weighting | The final strategy is selected inside the EBM notebook by validation PR-AUC, without changing the exported dataset. |
| `04_bilstm_pipeline` | Balanced batches and focal loss | Sequence data are not synthetically oversampled; focal-loss parameters and balanced-batch settings are handled inside the BiLSTM notebook. |

This notebook therefore performs only deterministic data preparation, filtering, feature engineering, winsorization and export. It does not rebalance the data.


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Resolve local paths from the repository root or its notebooks/ directory.
from pathlib import Path
_WORKING_DIR = Path.cwd().resolve()
_REPO_ROOT = (
    _WORKING_DIR.parent
    if _WORKING_DIR.name == "notebooks"
    and (_WORKING_DIR.parent / "requirements.txt").is_file()
    else _WORKING_DIR
)
DATA_DIR = str(Path(os.environ.get("DATA_DIR") or _REPO_ROOT).expanduser().resolve())
RAW_CSV  = os.path.join(DATA_DIR, 'firmy_dataset.csv')

# -------------------------------------------------------------------------
# Veľkostné hranice SME
# -------------------------------------------------------------------------
# Základné kritérium:
#   zamestnanci < 250
#   a zároveň aktíva < 43 mil. EUR ALEBO tržby < 50 mil. EUR
#
# SME_FILTER_MODE:
#   "strict"       = vyžaduje dostupné údaje a aktíva aj tržby >= 5 000 EUR
#   "soft_missing" = zjemnená verzia: necháva firmy s chýbajúcim počtom
#                    zamestnancov alebo chýbajúcimi tržbami, ak sú inak
#                    ekonomicky aktívne podľa dostupných údajov
# -------------------------------------------------------------------------
MAX_EMPLOYEES    = 250
MAX_TOTAL_ASSETS = 43_000_000
MAX_TURNOVER     = 50_000_000
MIN_ACTIVITY     = 5_000

SME_FILTER_MODE = "soft_missing"   # odporúčané pre zachovanie vzorky; alternatíva: "strict"

WINSOR_LOWER = 0.05
WINSOR_UPPER = 0.95

TRAIN_YEARS = list(range(2013, 2017))
VALID_YEARS = [2017]
TEST_YEARS  = [2018, 2019]

print(f"Data dir: {DATA_DIR}")
print(f"CSV exists: {os.path.exists(RAW_CSV)}")
print(f"SME_FILTER_MODE: {SME_FILTER_MODE}")


## 1. Load the raw firm-year panel

Reads the source CSV with explicit column names (the file ships with three header rows that are skipped), tries several common encodings, coerces numeric columns, and reports the resulting shape, the list of years present, the number of unique firms, and the per-row vs per-firm bankruptcy rate.

In [ ]:
column_names = [
    'id', 'rok', 'target', 'sk_nace', 'pocet_zamestnancov', 'pravna_forma',
    'celkove_aktiva', 'dlhodoby_nehmotny_majetok', 'dlhodoby_hmotny_majetok',
    'obezne_aktiva', 'zasoby', 'obchodne_pohladavky', 'peniaze',
    'vlastne_imanie', 'zakladne_imanie', 'nerozdeleny_zisk', 'celkove_zavazky',
    'dlhodobe_zavazky', 'dlhodobe_obch_zavazky', 'dlhodobe_danove_dlhy',
    'kratkodobe_zavazky', 'obchodne_zavazky', 'kratkodobe_danove_dlhy',
    'kratkodobe_financne_vypomoci', 'bankove_uvery_dlhodobe',
    'bankove_uvery_kratkodobe', 'celkove_trzby', 'trzby_spolu', 'odpisy',
    'vysledok_hospodarenia_z_hospodarskej_cinnosti',
    'trzby_z_predaja_cennych_papierov_a_podielov',
    'predane_cenne_papiere_a_podiely', 'vynosy_z_dhodobeho_financneho_majetku',
    'vynosy_z_kratkodobeho_financneho_majetku',
    'naklady_na_kratkodoby_financny_majetok',
    'vynosy_z_precenenia_cennych_papierov_a_vynosy_z_derivatovych_operacii',
    'naklady_na_precenenie_cennych_papierov_a_naklady_na_derivatove_operacie',
    'tvorba_a_zauctovanie_opravnych_poloziek_k_financnemu_majetku',
    'vynosove_uroky', 'nakladove_uroky', 'kurzove_zisky', 'kurzove_straty',
    'ostatne_vynosy_z_financnej_cinnosti',
    'ostatne_naklady_na_financnu_cinnost', 'celkove_dane', 'ebt',
    'cisty_zisk_strata', 'datum_vzniku'
]

TEXT_COLS = {'id', 'sk_nace', 'pravna_forma', 'datum_vzniku'}

# Načítanie s viacerými kódovaním
df = None
for enc in ('utf-8-sig', 'utf-8', 'cp1250', 'latin-1'):
    try:
        df = pd.read_csv(RAW_CSV, skiprows=3, names=column_names, low_memory=False, encoding=enc)
        print(f"Načítané s kódovaním: {enc}")
        break
    except Exception:
        continue

if df is None:
    raise ValueError("Nepodarilo sa načítať CSV")

# ── Konverzia číselných stĺpcov ────────────────────────────────────────
for col in column_names:
    if col not in TEXT_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values(['id', 'rok']).reset_index(drop=True)

print(f"Shape: {df.shape[0]:,} × {df.shape[1]}")
print(f"Roky: {sorted(df['rok'].dropna().unique().astype(int))}")
print(f"Firmy: {df['id'].nunique():,}")
print(f"\nTarget distribúcia (per riadok):")
print(df['target'].value_counts(dropna=False))
print(f"Target rate (per riadok): {df['target'].mean()*100:.3f}%")
print(f"Target rate (per firma):  {df.groupby('id')['target'].max().mean()*100:.3f}%")
print("\nNOTA: 0.2% per riadok je správna hodnota pre panel dáta (každá firma má viacero riadkov)")


## 2. Compute financial ratios

Uses a `safe_div` helper to compute liquidity, leverage, profitability, activity and interest-coverage ratios from raw accounting items. Extreme values of `interest_coverage` and `debt_ebitda` are clipped to plausible ranges before the later winsorization step. The resulting list of ratio columns is stored in `fin_ratio_cols` and reused by the lag/diff and export blocks.

In [ ]:
def safe_div(num, denom):
    num = pd.to_numeric(num, errors='coerce')
    denom = pd.to_numeric(denom, errors='coerce')
    result = np.where((denom != 0) & denom.notna() & num.notna(), num / denom, np.nan)
    return pd.Series(result, index=num.index if hasattr(num, 'index') else None)

# ── Likvidita ───────────────────────────────────────────────
df['cash_ta']       = safe_div(df['peniaze'], df['celkove_aktiva'])
df['current_ratio'] = safe_div(df['obezne_aktiva'], df['kratkodobe_zavazky'])
df['quick_ratio']   = safe_div(df['obezne_aktiva'] - df['zasoby'], df['kratkodobe_zavazky'])

# ── Solventnosť / Zadlženie ─────────────────────────────────
df['eq_ta']          = safe_div(df['vlastne_imanie'], df['celkove_aktiva'])
df['ltd_ta']         = safe_div(df['dlhodobe_zavazky'], df['celkove_aktiva'])
df['bl_ta']          = safe_div(df['bankove_uvery_dlhodobe'] + df['bankove_uvery_kratkodobe'], df['celkove_aktiva'])
df['debt_to_equity'] = safe_div(df['celkove_zavazky'], df['vlastne_imanie'])
df['debt_ratio']     = safe_div(df['celkove_zavazky'], df['celkove_aktiva'])

# Krytie dlhodobého majetku vlastným imaním + dlhodobými záväzkami
fixed_assets = df['dlhodoby_nehmotny_majetok'].fillna(0) + df['dlhodoby_hmotny_majetok'].fillna(0)
df['fixed_asset_coverage'] = safe_div(df['vlastne_imanie'] + df['dlhodobe_zavazky'], fixed_assets.replace(0, np.nan))

# Podiel krátkodobých záväzkov na celkových záväzkoch
df['short_debt_ratio'] = safe_div(df['kratkodobe_zavazky'], df['celkove_zavazky'])

# ── Rentabilita ─────────────────────────────────────────────
df['roa1'] = safe_div(df['cisty_zisk_strata'], df['celkove_aktiva'])
df['re_ta'] = safe_div(df['nerozdeleny_zisk'], df['celkove_aktiva'])
df['roe']   = safe_div(df['cisty_zisk_strata'], df['vlastne_imanie'])
df['ros']   = safe_div(df['cisty_zisk_strata'], df['celkove_trzby'])
df['roi']   = safe_div(df['ebt'], df['celkove_aktiva'])

# ── EBITDA ──────────────────────────────────────────────────
df['ebitda']        = df['vysledok_hospodarenia_z_hospodarskej_cinnosti'].fillna(0) + df['odpisy'].fillna(0)
df['ebitda_margin'] = safe_div(df['ebitda'], df['celkove_trzby'])
df['debt_ebitda']   = safe_div(df['celkove_zavazky'], df['ebitda'].replace(0, np.nan))

# ── Aktivita ────────────────────────────────────────────────
df['asset_turnover']  = safe_div(df['celkove_trzby'], df['celkove_aktiva'])
df['creditors_days']  = safe_div(365 * df['obchodne_zavazky'], df['celkove_trzby'])
df['receivables_days']= safe_div(365 * df['obchodne_pohladavky'], df['celkove_trzby'])

# ── Úrokové krytie ──────────────────────────────────────────
df['interest_coverage'] = safe_div(df['ebitda'], df['nakladove_uroky'].replace(0, np.nan))

# Clip extrémnych hodnôt pred winsorizáciou
df['interest_coverage'] = df['interest_coverage'].clip(-100, 100)
df['debt_ebitda']       = df['debt_ebitda'].clip(-50, 50)

fin_ratio_cols = [
    'cash_ta', 'eq_ta', 'ltd_ta', 'bl_ta', 'roa1', 're_ta', 'roe', 'ros', 'roi',
    'current_ratio', 'quick_ratio', 'debt_to_equity', 'debt_ratio', 'debt_ebitda',
    'fixed_asset_coverage', 'short_debt_ratio',
    'ebitda_margin', 'interest_coverage', 'asset_turnover', 'creditors_days', 'receivables_days',
    'ebitda'
]
fin_ratio_cols = [c for c in fin_ratio_cols if c in df.columns]

print(f"Finančné pomery: {len(fin_ratio_cols)}")
print(df[fin_ratio_cols[:8]].describe().round(3).to_string())


## 3. Non-financial features, industry dummies and macro controls

Derives firm age and `age_log` from `datum_vzniku`, builds size dummies from headcount, marks joint-stock companies, and turns the textual SK NACE category into one-hot industry dummies. Annual macroeconomic indicators (real GDP growth, unemployment, ECB refi rate, Euribor 3M, HICP, debt-to-GDP, nominal wage growth) are merged on `rok`.

In [ ]:
# ── Vek firmy ──────────────────────────────────────────────
def parse_date_safe(s):
    if pd.isna(s) or str(s).strip() == '':
        return pd.NaT
    for fmt in ('%d%b%Y', '%Y-%m-%d', '%d.%m.%Y', '%d/%m/%Y'):
        try:
            return pd.to_datetime(str(s), format=fmt, errors='coerce')
        except Exception:
            pass
    return pd.to_datetime(str(s), errors='coerce')

df['datum_vznik_dt'] = df['datum_vzniku'].apply(parse_date_safe)
df['year_of_creation'] = df['datum_vznik_dt'].dt.year
df['age'] = (df['rok'] - df['year_of_creation']).clip(lower=0)
df['age_log'] = np.log1p(df['age'])

# ── Právna forma (joint-stock = a.s.) ──────────────────────
df['joint_stock'] = (
    df['pravna_forma'].astype(str).str.contains('Akciov', case=False, na=False)
).astype(int)

# ── Veľkostné kategórie ─────────────────────────────────────
df['micro_company']  = ((df['pocet_zamestnancov'] <  10) | df['pocet_zamestnancov'].isna()).astype(int)
df['small_company']  = ((df['pocet_zamestnancov'] >= 10) & (df['pocet_zamestnancov'] < 50)).astype(int)
df['medium_company'] = ((df['pocet_zamestnancov'] >= 50) & (df['pocet_zamestnancov'] < 250)).astype(int)

# ── Odvetvová príslušnosť – dummy premenné z textovej kategórie ─────────────
# V tomto datasete je stĺpec sk_nace uložený ako textový názov odvetvia
# (napr. Stavebníctvo, Veľkoobchod), nie ako číselný SK NACE kód.
# Preto NEberieme prvé dva znaky, ale celý názov kategórie.

import re
import unicodedata

def slugify_industry(x):
    if pd.isna(x) or str(x).strip() == '':
        return 'missing'
    s = str(x).strip()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^0-9a-zA-Z]+', '_', s).strip('_').lower()
    return s if s else 'missing'

df['industry_group'] = df['sk_nace'].astype(str).str.strip()
df['industry_slug'] = df['industry_group'].apply(slugify_industry)

industry_dummies = pd.get_dummies(df['industry_slug'], prefix='industry', dtype=int)
df = pd.concat([df, industry_dummies], axis=1)

industry_cols = [c for c in df.columns if c.startswith('industry_')]

# Kompatibilita so staršími exportmi/notebookmi
nace_cols = industry_cols

# ── Makroekonomické ukazovatele (NBS/Eurostat) ──────────────
macro_df = pd.DataFrame({
    'rok': [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019],
    'hdp_zmena_pct':          [2.6,  1.6,  0.7,  2.7,  5.2,  1.9,  2.9,  4.1,  2.3],
    'nezamestnanost_pct':     [13.6, 13.9, 14.2, 13.2, 11.5,  9.6,  8.1,  6.5,  5.8],
    'zamestnanost_esa_zmena': [1.8,  0.1, -0.8,  1.4,  2.0,  2.4,  2.2,  2.0,  1.0],
    'mzda_nominal_zmena':     [2.2,  2.4,  2.4,  4.1,  2.9,  3.3,  4.6,  6.2,  3.8],
    'inflacia_hicp_pct':      [4.1,  3.7,  1.5, -0.1, -0.3, -0.5,  1.4,  2.5,  2.8],
    'ecb_refinancna_sadzba':  [1.00, 0.75, 0.25, 0.05, 0.05, 0.00, 0.00, 0.00, 0.00],
    'euribor_3m_priemer':     [1.39, 0.57, 0.22, 0.21,-0.02,-0.26,-0.33,-0.32,-0.36],
    'dlh_vs_hdp_pct':         [43.3, 51.7, 54.6, 53.4, 51.6, 52.1, 51.4, 49.3, 48.0],
})
df = df.merge(macro_df, on='rok', how='left')
macro_features = [c for c in macro_df.columns if c != 'rok']

nonfin_features = ['age', 'age_log', 'joint_stock', 'micro_company', 'small_company', 'medium_company']
print(f"Industry dummies: {len(industry_cols)}")
print(f"Makro features: {len(macro_features)}")
print(f"Dataset shape: {df.shape}")


## 4. SME filter

Applies the EU SME definition (employees < 250, and either total assets < 43 M EUR or turnover < 50 M EUR) plus an activity floor of 5 000 EUR. Two modes are supported: `strict` (requires non-missing employees / turnover) and `soft_missing` (used here — keeps rows with missing employee count or turnover as long as the remaining size criteria pass). A diagnostic block reports the share of rows that satisfy each individual sub-criterion.

In [ ]:
# =============================================================================
# 4. SME + activity filter
# =============================================================================
# Zjemnenie filtra:
# - základná definícia SME zostáva zachovaná:
#   zamestnanci < 250 a zároveň aktíva < 43 mil. EUR alebo tržby < 50 mil. EUR;
# - pri režime soft_missing sa nevyradia firmy len preto, že chýba údaj o počte
#   zamestnancov alebo tržbách, pokiaľ spĺňajú ostatné dostupné kritériá.
# - tým sa znižuje riziko, že RNN/EBM prídu o veľkú časť pozitívnej triedy.
# =============================================================================

n_before = len(df)
n_fail_before = df['target'].sum()

for col in ['pocet_zamestnancov', 'celkove_aktiva', 'celkove_trzby', 'trzby_spolu']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Tržby na účely filtra: primárne celkove_trzby, záložne trzby_spolu
turnover_for_filter = df['celkove_trzby'].copy()
if 'trzby_spolu' in df.columns:
    turnover_for_filter = turnover_for_filter.fillna(df['trzby_spolu'])

if SME_FILTER_MODE == "strict":
    employee_ok = df['pocet_zamestnancov'] < MAX_EMPLOYEES
    financial_size_ok = (
        (df['celkove_aktiva'] < MAX_TOTAL_ASSETS) |
        (turnover_for_filter < MAX_TURNOVER)
    )
    active_ok = (
        (df['celkove_aktiva'] >= MIN_ACTIVITY) &
        (turnover_for_filter >= MIN_ACTIVITY)
    )

elif SME_FILTER_MODE == "soft_missing":
    employee_ok = (
        (df['pocet_zamestnancov'] < MAX_EMPLOYEES) |
        df['pocet_zamestnancov'].isna()
    )
    financial_size_ok = (
        (df['celkove_aktiva'] < MAX_TOTAL_ASSETS) |
        (turnover_for_filter < MAX_TURNOVER)
    )

    # Aktivitu vyžadujeme podľa aktív; pri tržbách ponecháme pozorovania s missing,
    # aby sme zbytočne nevyradili firmy iba pre nevyplnený údaj.
    active_ok = (
        (df['celkove_aktiva'] >= MIN_ACTIVITY) &
        ((turnover_for_filter >= MIN_ACTIVITY) | turnover_for_filter.isna())
    )

else:
    raise ValueError("SME_FILTER_MODE musí byť 'strict' alebo 'soft_missing'.")

sme_mask = employee_ok & financial_size_ok & active_ok
df = df[sme_mask].copy().reset_index(drop=True)

n_after = len(df)
n_fail_after = df['target'].sum()

print(f"SME filter ({SME_FILTER_MODE}): {n_before:,} → {n_after:,} riadkov (- {n_before-n_after:,})")

if n_fail_before > 0:
    print(f"Bankroty:   {n_fail_before:.0f} → {n_fail_after:.0f} (zachovaných: {n_fail_after/n_fail_before*100:.1f}%)")
else:
    print(f"Bankroty:   {n_fail_before:.0f} → {n_fail_after:.0f}")

print(f"Target rate: {df['target'].mean()*100:.3f}%")

print("\nDistribúcia targetu podľa rokov:")
yr = df.groupby('rok')['target'].agg(['sum','count','mean'])
yr.columns = ['failures','total','rate']
yr['rate'] = yr['rate'].mul(100).round(3)
print(yr.to_string())

print("\nDiagnostika vyradenia podľa komponentov filtra:")
print(f"  employee_ok:        {employee_ok.mean()*100:.2f}%")
print(f"  financial_size_ok:  {financial_size_ok.mean()*100:.2f}%")
print(f"  active_ok:          {active_ok.mean()*100:.2f}%")


## 5. Winsorization (computed on training years only)

For every financial ratio, the 5th and 95th percentile bounds are estimated **exclusively on the training years 2013–2016** and then applied to the entire panel (train, validation, test). This prevents leakage of distributional information from the held-out years.

In [ ]:
# Winsorizácia sa vypočíta VÝHRADNE z tréningových rokov (2013–2016)
# a aplikuje na celý dataset → prevencia úniku informácií z validácie/testu

train_mask = df['rok'].isin(TRAIN_YEARS)

winsor_bounds = {}
for col in fin_ratio_cols:
    if col not in df.columns:
        continue
    train_vals = df.loc[train_mask, col].dropna()
    if len(train_vals) < 10:
        continue
    lo = train_vals.quantile(WINSOR_LOWER)
    hi = train_vals.quantile(WINSOR_UPPER)
    winsor_bounds[col] = (lo, hi)
    df[col] = df[col].clip(lo, hi)

print(f"Winsorizácia dokončená: {len(winsor_bounds)} stĺpcov na [{WINSOR_LOWER*100:.0f}. – {WINSOR_UPPER*100:.0f}. percentile]")
print("(hranice vypočítané z trénu 2013-2016, aplikované na všetky roky)")


## 6. Lag and difference features for the EBM

Following Maturo et al. (2025), the EBM has no temporal memory: every row is treated independently. To inject economically meaningful trend information without moving to a sequence model, we add `lag1`, `lag2` and the corresponding `diff1`, `diff2` for the main ratios. Lags are computed within each firm (`groupby('id').shift(...)`) on a `(id, rok)`-sorted frame.

In [ ]:
# Zoradenie pre správny výpočet lagov
df = df.sort_values(['id', 'rok']).reset_index(drop=True)

# Finančné pomery pre lag features (tie ktoré majú zmysluplnú medziroč. dynamiku)
lag_ratio_cols = [
    'cash_ta', 'eq_ta', 'ltd_ta', 'roa1', 're_ta', 'roe', 'ros',
    'current_ratio', 'quick_ratio', 'debt_ratio', 'debt_ebitda',
    'ebitda_margin', 'interest_coverage', 'asset_turnover',
]
lag_ratio_cols = [c for c in lag_ratio_cols if c in df.columns]

for col in lag_ratio_cols:
    # Lag-1 (hodnota spred 1 roku)
    df[f'{col}_lag1'] = df.groupby('id')[col].shift(1)
    # Lag-2 (hodnota spred 2 rokov)
    df[f'{col}_lag2'] = df.groupby('id')[col].shift(2)
    # Diferencie
    df[f'{col}_diff1'] = df[col] - df[f'{col}_lag1']
    df[f'{col}_diff2'] = df[col] - df[f'{col}_lag2']

lag_cols = [c for c in df.columns if any(c.endswith(s) for s in ['_lag1','_lag2','_diff1','_diff2'])]
print(f"Lag/diff features vytvorené: {len(lag_cols)}")
print(f"Príklad pre 'roa1':")
print(df[['id','rok','roa1','roa1_lag1','roa1_diff1','roa1_diff2']].head(8).to_string(index=False))


## 7. Export `data_logit.csv` (and `data_logit_strict.csv` alias)

Builds a single rich logit export containing both the engineered ratios used by the benchmark / enhanced logit specifications and the raw accounting items required by the strict Káčer & Alexy-style notebook (which recomputes ratios itself). Compatibility aliases `sknace` and `EBT` are added. Engineered ratios are imputed with **training-period medians** (computed only on 2013–2016) and the medians are persisted to `logit_train_medians.pkl`. Raw accounting items are left untouched so the strict notebook can handle missingness on its own terms.

In [ ]:
id_cols = ['id', 'rok', 'target']

# -------------------------------------------------------------------------
# LOGIT EXPORT
# -------------------------------------------------------------------------
# Cieľ:
# 1. zachovať kompatibilitu s pôvodným benchmark / enhanced logit pipelineom,
# 2. zároveň vyexportovať surové stĺpce potrebné pre strict Káčer & Alexy logit,
#    aby sa pomery dali nanovo dopočítať bez chýb typu chýbajúci datum_vzniku.
#
# Export preto obsahuje:
# - engineered logit features,
# - nefinančné + NACE + makro premenné,
# - surové účtovné položky pre strict logit,
# - aliasy názvov stĺpcov (`sknace`, `EBT`) kvôli kompatibilite notebookov.
# -------------------------------------------------------------------------

# 1) Engineered logit features používané v benchmark/enhanced pipelineoch
logit_ratio_cols = [
    'cash_ta', 'eq_ta', 'ltd_ta', 'bl_ta', 'roa1', 're_ta', 'roe', 'ros',
    'current_ratio', 'quick_ratio', 'debt_to_equity', 'ebitda_margin',
    'interest_coverage', 'asset_turnover', 'creditors_days'
]
logit_ratio_cols = [c for c in logit_ratio_cols if c in df.columns]

# 2) Surové účtovné položky potrebné pre strict Káčer & Alexy-style logit
strict_raw_cols = [
    'sk_nace', 'pocet_zamestnancov', 'pravna_forma', 'datum_vzniku',
    'celkove_aktiva', 'obezne_aktiva', 'zasoby', 'peniaze', 'obchodne_pohladavky',
    'vlastne_imanie', 'nerozdeleny_zisk',
    'celkove_zavazky', 'dlhodobe_zavazky', 'kratkodobe_zavazky',
    'obchodne_zavazky', 'bankove_uvery_dlhodobe', 'bankove_uvery_kratkodobe',
    'celkove_trzby', 'trzby_spolu', 'odpisy', 'ebt', 'nakladove_uroky',
    'vynosove_uroky', 'cisty_zisk_strata'
]
strict_raw_cols = [c for c in strict_raw_cols if c in df.columns]

# 3) Spoločný export
logit_cols = (
    id_cols
    + strict_raw_cols
    + logit_ratio_cols
    + nonfin_features
    + nace_cols
    + macro_features
)
logit_cols = list(dict.fromkeys([c for c in logit_cols if c in df.columns]))

logit_df = df[logit_cols].copy()

# Aliasy názvov stĺpcov pre strict notebook
if 'sk_nace' in logit_df.columns and 'sknace' not in logit_df.columns:
    logit_df['sknace'] = logit_df['sk_nace']
if 'ebt' in logit_df.columns and 'EBT' not in logit_df.columns:
    logit_df['EBT'] = logit_df['ebt']

# ── Mediánová imputácia LEN pre engineered pomery z tréningovej vzorky ─────
# Raw účtovné položky nechávame bez imputácie, aby strict logit vedel sám
# transparentne dopočítať pomery a ošetriť missing podľa svojej metodiky.
train_mask_logit = logit_df['rok'].isin(TRAIN_YEARS)
impute_cols = [c for c in logit_ratio_cols if c in logit_df.columns and logit_df[c].isnull().sum() > 0]

train_medians = {}
for col in impute_cols:
    med = logit_df.loc[train_mask_logit, col].median()
    train_medians[col] = med
    logit_df[col] = logit_df[col].fillna(med)

print(f"Imputované engineered pomery ({len(impute_cols)}) — mediány z tréningu 2013–2016:")
for col, med in train_medians.items():
    nan_before = df[col].isnull().sum()
    nan_pct = nan_before / len(df) * 100
    print(f"  {col:<22}: medián={med:.4f}  (pôvodné NaN: {nan_pct:.1f}%)")

# Nefinančné, makro a NACE dummies ponecháme; kritické missing reportujeme
aux_cols = [c for c in (nonfin_features + macro_features) if c in logit_df.columns]
aux_missing = logit_df[aux_cols].isnull().mean().sort_values(ascending=False)
aux_missing = aux_missing[aux_missing > 0]
if len(aux_missing):
    print("\nPomocné stĺpce s missing hodnotami:")
    print((aux_missing * 100).round(2).to_string())
else:
    print("\nPomocné stĺpce bez missing hodnôt.")

# Uloženie mediánov pre benchmark/enhanced logit pipeline
import pickle as _pkl
with open(os.path.join(DATA_DIR, 'logit_train_medians.pkl'), 'wb') as _f:
    _pkl.dump(train_medians, _f)
print("Mediány uložené: logit_train_medians.pkl")

# Hlavný export — bohatý logit dataset
logit_path = os.path.join(DATA_DIR, 'data_logit.csv')
logit_df.to_csv(logit_path, index=False)

# Voliteľný alias exportu pre strict logit, rovnaký obsah
logit_strict_path = os.path.join(DATA_DIR, 'data_logit_strict.csv')
logit_df.to_csv(logit_strict_path, index=False)

print(f"\nUložené: {logit_path}")
print(f"Alias :  {logit_strict_path}")
print(f"Shape: {logit_df.shape} | Target rate: {logit_df['target'].mean()*100:.3f}%")
print(f"Engineered logit features: {[c for c in logit_ratio_cols if c in logit_df.columns]}")
print(f"Strict raw cols: {[c for c in strict_raw_cols if c in logit_df.columns]}")
print(f"Aliasy pridané: {[c for c in ['sknace', 'EBT'] if c in logit_df.columns]}")


## 8. Export `data_ebm.csv`

Combines raw accounting items, engineered ratios, the lag/diff block, non-financials, industry dummies and macro features. Remaining `NaN` values are replaced with the numeric sentinel `-999`. This does not automatically invoke EBM's native missing-value handling; inspect the fitted bins and contributions. The dataset has the same panel size as the logit export (target rate ≈ 0.184 %).

In [ ]:
# EBM dataset: surové absolútne hodnoty + finančné pomery + lag/diff + nefinančné + NACE + makro
# NaN sa nahrádza -999 (EBM vie pracovať so špeciálnou missing hodnotou)

ebm_raw_cols = [
    'pocet_zamestnancov', 'celkove_aktiva', 'obezne_aktiva', 'zasoby',
    'obchodne_pohladavky', 'peniaze', 'vlastne_imanie', 'zakladne_imanie',
    'nerozdeleny_zisk', 'celkove_zavazky', 'dlhodobe_zavazky',
    'kratkodobe_zavazky', 'obchodne_zavazky', 'kratkodobe_financne_vypomoci',
    'bankove_uvery_dlhodobe', 'bankove_uvery_kratkodobe',
    'celkove_trzby', 'trzby_spolu', 'odpisy',
    'vysledok_hospodarenia_z_hospodarskej_cinnosti',
    'vynosove_uroky', 'nakladove_uroky', 'cisty_zisk_strata', 'ebt'
]
ebm_raw_cols = [c for c in ebm_raw_cols if c in df.columns]

ebm_cols = (id_cols + ebm_raw_cols + fin_ratio_cols + lag_cols
            + nonfin_features + nace_cols + macro_features)
ebm_cols = list(dict.fromkeys([c for c in ebm_cols if c in df.columns]))  # unique, zachovanie poradia

ebm_df = df[ebm_cols].copy()

# Konverzia všetkých numeric stĺpcov (nie id)
for col in ebm_df.columns:
    if col not in ['id']:
        ebm_df[col] = pd.to_numeric(ebm_df[col], errors='coerce')

# fillna(-999) LEN pre numerické stĺpce
num_cols = ebm_df.select_dtypes(include='number').columns
ebm_df[num_cols] = ebm_df[num_cols].fillna(-999)

ebm_path = os.path.join(DATA_DIR, 'data_ebm.csv')
ebm_df.to_csv(ebm_path, index=False)
print(f"Uložené: {ebm_path}")
print(f"Shape: {ebm_df.shape} | Target rate: {ebm_df['target'].mean()*100:.3f}%")
print(f"Lag/diff features v EBM datasete: {sum(1 for c in ebm_df.columns if '_lag' in c or '_diff' in c)}")
print(f"NaN zostávajúce: {ebm_df.isnull().sum().sum()}")


## 9. Export `data_rnn.csv`

Keeps only firm-years that belong to a run of at least two consecutive years (`seq_len >= 2`); the sequence length is computed by `compute_consecutive_years`. Sequence-level ratio columns are paired with their lag/diff counterparts. Static features (non-financials, macro, industry one-hots) live in a separate block. Missing values in modeled features are filled with **training-period medians**, persisted to `rnn_train_medians.pkl`. The exported frame is saved with `utf-8-sig` so Excel opens it without garbled diacritics.

In [ ]:
# =============================================================================
# 9. Export: data_rnn.csv
# =============================================================================
# Cieľ:
# - exportovať sekvenčné finančné premenné vrátane lag/diff trendových premenných,
# - pridať odvetvové dummy premenné do statickej vetvy RNN,
# - nepoužiť -999 pre RNN vstupy; namiesto toho použiť mediánovú imputáciu
#   vypočítanú výlučne z tréningových rokov 2013–2016.
# =============================================================================

# Základné level finančné premenné pre sekvenčnú vetvu
rnn_seq_level_cols = logit_ratio_cols + ['ebitda']
rnn_seq_level_cols = [c for c in rnn_seq_level_cols if c in df.columns]

# Explicitné dynamické premenné pre sekvenčnú vetvu
# Vznikli v bunke s lag/diff features.
rnn_dynamic_cols = [
    c for c in lag_cols
    if c in df.columns and any(c.endswith(s) for s in ['_lag1', '_lag2', '_diff1', '_diff2'])
]

# Statické premenné: nefinančné + makro + odvetvové dummy
industry_export_cols = []
if 'industry_cols' in globals():
    industry_export_cols = [c for c in industry_cols if c in df.columns]
elif 'nace_cols' in globals():
    industry_export_cols = [c for c in nace_cols if c in df.columns]

rnn_static_cols = nonfin_features + macro_features + industry_export_cols
rnn_static_cols = [c for c in rnn_static_cols if c in df.columns]

rnn_feature_cols = list(dict.fromkeys(rnn_seq_level_cols + rnn_dynamic_cols + rnn_static_cols))
rnn_cols = list(dict.fromkeys([c for c in (id_cols + rnn_feature_cols) if c in df.columns]))

rnn_df = df[rnn_cols].copy().sort_values(['id', 'rok']).reset_index(drop=True)

# Počet po sebe nasledujúcich rokov pre každú firmu
def compute_consecutive_years(grp):
    grp = grp.sort_values('rok')
    years = grp['rok'].values
    seq = [1]
    for i in range(1, len(years)):
        seq.append(seq[-1] + 1 if years[i] == years[i-1] + 1 else 1)
    return pd.Series(seq, index=grp.index)

rnn_df['seq_len'] = rnn_df.groupby('id', group_keys=False).apply(compute_consecutive_years)
rnn_df = rnn_df[rnn_df['seq_len'] >= 2].copy().reset_index(drop=True)

# -------------------------------------------------------------------------
# Numerická konverzia + mediánová imputácia z tréningovej vzorky
# -------------------------------------------------------------------------
num_feature_cols = [c for c in rnn_feature_cols if c in rnn_df.columns]

for col in num_feature_cols:
    rnn_df[col] = pd.to_numeric(rnn_df[col], errors='coerce')

rnn_df[num_feature_cols] = rnn_df[num_feature_cols].replace([np.inf, -np.inf], np.nan)

train_mask_rnn = rnn_df['rok'].isin(TRAIN_YEARS)

rnn_train_medians = {}
for col in num_feature_cols:
    if rnn_df[col].isna().sum() == 0:
        continue

    med = rnn_df.loc[train_mask_rnn, col].median()
    if pd.isna(med):
        med = 0.0

    rnn_train_medians[col] = float(med)
    rnn_df[col] = rnn_df[col].fillna(med)

# Kontrola, či zostali NaN len mimo modelových stĺpcov
remaining_nan = int(rnn_df[num_feature_cols].isna().sum().sum())

# Uloženie mediánov pre reprodukovateľnosť
import pickle as _pkl
with open(os.path.join(DATA_DIR, 'rnn_train_medians.pkl'), 'wb') as _f:
    _pkl.dump(rnn_train_medians, _f)

rnn_path = os.path.join(DATA_DIR, 'data_rnn.csv')
rnn_df.to_csv(rnn_path, index=False, encoding='utf-8-sig')

print(f"Uložené: {rnn_path}")
print(f"Shape: {rnn_df.shape} | Target rate: {rnn_df['target'].mean()*100:.3f}%")
print(f"Sekvenčné level features: {len(rnn_seq_level_cols)}")
print(f"Dynamické lag/diff features: {len(rnn_dynamic_cols)}")
print(f"Statické features spolu: {len(rnn_static_cols)}")
print(f"  z toho industry dummy: {len(industry_export_cols)}")
print(f"Imputované RNN features: {len(rnn_train_medians)}")
print(f"NaN v modelových RNN features po imputácii: {remaining_nan}")

print("\nDistribúcia dĺžok sekvencií:")
print(rnn_df['seq_len'].value_counts().sort_index().to_string())

print("\nPríklady dynamických features v data_rnn.csv:")
print(rnn_dynamic_cols[:20])


## 10. Final summary

Reloads each export and prints row counts per split (train / valid / test), the per-row target rate, and the column count. Provides a sanity check that the logit, EBM and RNN datasets all share the same set of firm-years (modulo the RNN-only `seq_len ≥ 2` filter).

In [ ]:
print("=" * 90)
print(f"{'Dataset':<18} {'Riadky':>10} {'Train':>10} {'Valid':>8} {'Test':>8} {'Target%':>8}")
print("-" * 90)
for name, path in [
    ('Logit', logit_path),
    ('Logit strict alias', logit_strict_path),
    ('EBM', ebm_path),
    ('RNN', rnn_path)
]:
    tmp = pd.read_csv(path, nrows=None, low_memory=False)
    tr = tmp[tmp['rok'].between(2013, 2016)]
    va = tmp[tmp['rok'] == 2017]
    te = tmp[tmp['rok'].between(2018, 2019)]
    print(f"{name:<18} {len(tmp):>10,} {len(tr):>10,} {len(va):>8,} {len(te):>8,} {tmp['target'].mean()*100:>7.3f}%")

print("=" * 90)
print("\nSúbory pripravené pre modely:")
for n, p in [
    ('Logit', logit_path),
    ('Logit strict alias', logit_strict_path),
    ('EBM', ebm_path),
    ('RNN', rnn_path)
]:
    tmp = pd.read_csv(p, nrows=0)
    print(f"  {n}: {p} ({len(tmp.columns)} stĺpcov)")

print("\nPoznámky:")
print("- data_logit.csv teraz obsahuje aj surové účtovné položky + aliasy sknace/EBT")
print("- strict logit notebook môže čítať data_logit.csv aj data_logit_strict.csv")
print("- EBM dataset obsahuje lag-1, lag-2 a diff features pre temporálny kontext")


## Changelog / GitHub notes

- Added an explicit class-imbalance strategy block.
- Clarified that the exported `data_*.csv` files are not resampled.
- The model-specific imbalance treatment is documented in notebooks `01`, `03` and `04`.
